# Lab 1: Tokenisation and embeddings

In this lab, you will build an understanding of how text can be transformed into representations that computers can process and learn from. Specifically, you will explore two key concepts: *tokenisation* and *embeddings*. Tokenisation splits text into smaller units such as words, subwords, or characters. Embeddings are dense, fixed-size vector representations of tokens in a continuous space.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Part 1: Tokenisation

In the first part of the lab, you will code and analyse a tokeniser based on the Byte Pair Encoding (BPE) algorithm.

### Utility functions

The BPE tokeniser transforms text into a list of integers representing tokens. As a warm-up, you will implement two utility functions on such lists. To simplify things, we define a shorthand for the type of pairs of integers:

In [1]:
Pair = tuple[int, int]


#### 🧩 Task 1.01: Counting pairs

Write a function that counts all occurrences of pairs of consecutive token IDs in a given list. The function should return a dictionary that maps each pair to its count. Skip counts that are zero.

In [2]:
def count(ids: list[int]) -> dict[Pair, int]:
    """
    Count how many times each consecutive pair of token IDs appears.

    Example:
        ids = [1, 2, 3, 1, 2]

    Pairs:
        (1,2), (2,3), (3,1), (1,2)

    Returns:
        {
            (1,2): 2,
            (2,3): 1,
            (3,1): 1
        }
    """

    # Dictionary that maps a pair -> frequency count
    counts = {}

    # Iterate through all consecutive positions
    # We stop at len(ids) - 1 because we look at i and i+1
    for i in range(len(ids) - 1):

        # Create the consecutive pair
        pair = (ids[i], ids[i + 1])

        # If this pair has not been seen before,
        # initialize its count to zero
        if pair not in counts:
            counts[pair] = 0

        # Increment the count for this pair
        counts[pair] += 1

    return counts

#### 🧩 Task 1.02: Replacing pairs

Write a function that traverses a list of token IDs from left to right and replaces all occurrences of a specified pair of consecutive IDs by a new ID. The function should return the modified list.

In [3]:
def replace(ids: list[int], pair: Pair, new_id: int) -> list[int]:
    """
    Replace every left-to-right occurrence of a given consecutive pair
    with a new token ID.

    Example:
        ids = [1, 2, 3, 1, 2]
        pair = (1, 2)
        new_id = 99

    Result:
        [99, 3, 99]

    This is the basic merge operation used in BPE.
    """

    # This list will store the new sequence after replacements
    new_ids = []

    # We use a while loop because sometimes we need to skip two tokens
    i = 0

    # Traverse the list from left to right
    while i < len(ids):

        # Check whether there is a next token
        # and whether the current + next token match the target pair
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:

            # Replace the pair with the new merged token ID
            new_ids.append(new_id)

            # Skip both tokens because they have been merged
            i += 2

        else:
            # If no merge happens, keep the current token unchanged
            new_ids.append(ids[i])

            # Move to the next token
            i += 1

    return new_ids

### Encoding and decoding

The next cell contains the core code for the tokeniser in the form of a class `Tokenizer`. This class implements two methods: `encode()` converts an input text to a list of token IDs by exhaustively applying rules for merging pairs of consecutive IDs (stored in the dictionary `self.merges`), and `decode()` reverses this process.

**Note that the set of merge rules is initially empty; you will add rules in Task&nbsp;1.04.**

In [4]:
class Tokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(2**8)}

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while True:
            counts = count(ids)
            mergeable_pairs = counts.keys() & self.merges.keys()
            if len(mergeable_pairs) == 0:
                break
            to_merge = min(mergeable_pairs, key=self.merges.get)  # type: ignore
            ids = replace(ids, to_merge, self.merges[to_merge])
        return ids

    def decode(self, ids):
        return b"".join((self.vocab[i] for i in ids)).decode("utf-8")
    
    

class Tokenizer:
    def __init__(self) -> None:
        """
        Create an initially empty BPE tokenizer.

        At the beginning:
        - The vocabulary contains only the 256 possible byte values.
        - There are no merge rules yet.
        """

        # Dictionary mapping a pair of token IDs to the new merged token ID.
        #
        # Example:
        #   self.merges[(101, 114)] = 256
        #
        # This means:
        #   token 101 = byte for 'e'
        #   token 114 = byte for 'r'
        #   merge (101, 114) into new token 256
        self.merges: dict[Pair, int] = {}

        # Dictionary mapping each token ID to the byte sequence it represents.
        #
        # Initially, token 0 represents bytes([0]),
        # token 1 represents bytes([1]),
        # ...
        # token 255 represents bytes([255]).
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(2**8)}

    def encode(self, text: str) -> list[int]:
        """
        Convert a string into a list of token IDs.

        The process is:
        1. Convert the text into UTF-8 bytes.
        2. Treat each byte as one initial token.
        3. Repeatedly apply the learned BPE merge rules.
        4. Stop when no more merge rule applies.
        """

        # Convert text into raw UTF-8 bytes.
        #
        # Example:
        #   "er".encode("utf-8") gives b"er"
        #   list(b"er") gives [101, 114]
        ids: list[int] = list(text.encode("utf-8"))

        while True:
            # Count all adjacent token pairs in the current sequence.
            #
            # Example:
            #   [101, 114, 101] has pairs:
            #   (101, 114), (114, 101)
            counts: dict[Pair, int] = count(ids)

            # Find which pairs are both:
            # - present in the current text
            # - present in the learned merge rules
            mergeable_pairs = counts.keys() & self.merges.keys()

            # If no learned merge rule applies, encoding is finished.
            if len(mergeable_pairs) == 0:
                break

            # Choose the merge rule with the smallest assigned token ID.
            #
            # This means older/earlier learned merge rules are applied first.
            to_merge: Pair = min(mergeable_pairs, key=self.merges.get)  # type: ignore

            # Replace every occurrence of that pair with its merged token ID.
            ids = replace(ids, to_merge, self.merges[to_merge])

        return ids

    def decode(self, ids: list[int]) -> str:
        """
        Convert a list of token IDs back into a string.

        Each token ID is mapped back to its byte representation.
        The byte sequences are concatenated and decoded as UTF-8.
        """

        # Convert each token ID into bytes, concatenate all bytes,
        # then decode the result back into a Python string.
        return b"".join((self.vocab[i] for i in ids)).decode("utf-8")

#### 🎓 Task 1.03: Encoding and decoding

Explain how the code implements the BPE algorithm. Use the following steps to check your understanding:

**Step&nbsp;1.** Annotate the attributes and methods of the `Tokenizer` class with their Python types. In particular, what is the type of `self.merges`? Use the `Pair` shorthand. What does a merge rule look like?

**Step&nbsp;2.** Explain how the implementation chooses which merge rule to apply. Provide an example that illustrates the logic. Construct the example such that you get a different result when you use `max()` instead of `min()`.

##### Explanation

`self.merges` has type `dict[Pair, int]`, where `Pair = tuple[int, int]`.
Each entry maps a pair of existing token IDs to the ID of the new merged
token. For example, `(101, 114): 256` means that tokens 101 (`e`) and
114 (`r`) are merged into token 256 (`er`). The vocabulary `self.vocab`
has type `dict[int, bytes]`, while `encode()` maps a `str` to `list[int]`
and `decode()` maps `list[int]` back to a `str`.

During encoding, the tokenizer first finds all pairs that occur in the
current token sequence and that also have a learned merge rule. It then
selects the applicable rule with the smallest merge ID using

`min(mergeable_pairs, key=self.merges.get)`.

This preserves the order in which the BPE rules were learned. For example,
if `(1, 2): 256` and `(2, 3): 257` are both applicable to `[1, 2, 3]`,
`min()` merges `(1, 2)` and produces `[256, 3]`. Using `max()` instead
would merge `(2, 3)` and produce `[1, 257]`, resulting in a different
tokenisation.

### Training a tokeniser

Upon initialisation, a tokeniser has an empty set of merge rules. Your next task is to complete the BPE algorithm and write code to learn these merge rules from a text.

#### 🎓 Task 1.04: Training a tokeniser

Write a function that induces a BPE tokeniser from a given text. The function should take the text (a string) and a target vocabulary size as input and return the trained tokeniser.

In [10]:
def from_text(text: str, vocab_size: int) -> Tokenizer:
    """
    Train a byte-level BPE tokenizer on the given text.

    Parameters
    ----------
    text:
        Training text used to learn the merge rules.

    vocab_size:
        Desired total vocabulary size.

        Since the tokenizer starts with the 256 possible byte values,
        we need to learn:

            vocab_size - 256

        additional merged tokens.

    Returns
    -------
    Tokenizer
        A trained tokenizer containing the learned merge rules and vocabulary.
    """

    # Create a new tokenizer.
    # Initially:
    # - vocab contains the 256 possible byte values
    # - merges is empty
    tok = Tokenizer()

    # Convert the training text into UTF-8 bytes.
    #
    # Example:
    #   "hello".encode("utf-8") -> b"hello"
    #   list(...)               -> [104, 101, 108, 108, 111]
    #
    # These byte IDs are our initial tokens.
    ids = list(text.encode("utf-8"))

    # The tokenizer already contains 256 byte tokens.
    #
    # Therefore, if vocab_size = 300,
    # we need to learn 300 - 256 = 44 new tokens.
    num_merges = vocab_size - 256

    # Learn one new merge rule at a time.
    for i in range(num_merges):

        # Count how often every adjacent token pair occurs
        # in the CURRENT token sequence.
        counts = count(ids)

        # If fewer than two tokens remain, there are no pairs left to merge.
        if len(counts) == 0:
            break

        # Find the most frequent adjacent pair.
        #
        # This is the central BPE training step:
        # BPE repeatedly merges the pair that appears most often.
        pair = max(counts, key=counts.get)

        # Assign a new token ID.
        #
        # The first learned token gets ID 256,
        # the second gets 257, then 258, ...
        new_id = 256 + i

        # Store the merge rule.
        #
        # Example:
        #   (101, 114) -> 256
        tok.merges[pair] = new_id

        # Construct the byte representation of the new token.
        #
        # If:
        #   vocab[101] = b"e"
        #   vocab[114] = b"r"
        #
        # then:
        #   vocab[256] = b"er"
        tok.vocab[new_id] = tok.vocab[pair[0]] + tok.vocab[pair[1]]

        # Apply the newly learned merge to the training sequence.
        #
        # This is important because the next pair counts must be computed
        # on the UPDATED sequence, not the original byte sequence.
        ids = replace(ids, pair, new_id)

    return tok

To help you test your implementation, we provide three text files together with tokenisers trained on these files. Each text file contains the first 1&nbsp;million Unicode characters in a language-specific Wikipedia:

| Text file | Tokeniser file | Wikipedia |
|---|---|---|
| `wiki-en-1m.txt` | `wiki-en-1m.tok` | [Simple English](https://simple.wikipedia.org/) |
| `wiki-is-1m.txt` | `wiki-is-1m.tok` | [Icelandic](https://is.wikipedia.org/) |
| `wiki-sv-1m.txt` | `wiki-sv-1m.tok` | [Swedish](https://sv.wikipedia.org/) |

A tokeniser file consists of lines specifying merge rules. For example, the first line in the tokeniser file for Swedish is `101 114`, which expresses that this rule combines the token with ID 101 (`e`) and the token with ID 114 (`r`). The ID of the new token (`er`) is 256 plus the (zero-indexed) line number on which the rule is found. The following code saves a `Tokenizer` to a file with this format:

In [11]:
def save(tokenizer: Tokenizer, filename: str) -> None:
    with open(filename, "w") as f:
        for fst, snd in tokenizer.merges:
            print(f"{fst} {snd}", file=f)

In [15]:
# Read the English Wikipedia training text.
with open("wiki-en-1m.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Count how many merge rules are in the provided tokenizer.
with open("wiki-en-1m.tok", "r") as f:
    num_merges = sum(1 for _ in f)

# The vocabulary contains the initial 256 byte tokens
# plus one new token for every learned merge rule.
vocab_size = 256 + num_merges

# Train our tokenizer using the same target vocabulary size.
tok = from_text(text, vocab_size)

# Save our learned merge rules.
save(tok, "my-wiki-en-1m.tok")

In [16]:
!diff my-wiki-en-1m.tok wiki-en-1m.tok

To test your code, compare your saved tokeniser to the provided tokeniser using the `diff` command line tool.

**Note that training a tokeniser can take a few minutes.**

### STATHIS: Task 1.04 – Training BPE

The tokenizer starts with a vocabulary containing the 256 possible byte
values. To reach a target vocabulary size of `vocab_size`, we therefore
learn `vocab_size - 256` additional merge rules.

At each iteration, all adjacent token pairs in the current training sequence
are counted. The most frequent pair is selected using `max(counts,
key=counts.get)` and assigned the next available token ID, starting from 256.
The new token's byte representation is constructed by concatenating the byte
representations of the two tokens that were merged.

After learning a merge rule, all occurrences of that pair are replaced in the
training sequence. Pair frequencies are then recomputed on the updated
sequence, and the process repeats until the desired vocabulary size is reached.

### Tokenisation quirks

The tokeniser is a key component of language models, as it defines the minimal chunks of text the model can “see” and work with. As you will see in this section, tokenisation is also responsible for several deficiencies and unexpected behaviours of language models.

One helpful tool for experimenting with tokenisers in language models is the web app [Tiktokenizer](https://tiktokenizer.vercel.app/). This app lets you play around with, among others, [`o200k_base`](https://tiktokenizer.vercel.app/?model=o200k_base), the tokeniser used in ChatGPT&nbsp;5.2. You can also use OpenAI’s own [Tokenizer](https://platform.openai.com/tokenizer) page.

#### 🎓 Task 1.05: Tokenisation quirks

Prompt [ChatGPT](https://chatgpt.com/) to reverse the letters in the following words:

```
creativecommons
MERCHANTABILITY
NSNotification
authentication
```

How many of these words come out right? What happens when you modify the prompt and explicitly disable “thinking” and external tools? What could be the problem when words come out wrong? Generate ideas by inspecting the words in Tiktokenizer. Try to come up with other prompts that illustrate problems related to tokenisation.


### ANSWER

When I asked ChatGPT to reverse the four words normally, all **4/4** words
were reversed correctly. When I asked again while explicitly disabling
thinking and external tools, **3/4** were correct.

The incorrect case was `creativecommons`, which was reversed as
`snommocetaerc` instead of `snommocevitaerc`.

Inspecting the words with the `o200k_base` tokenizer shows that they are not
necessarily represented as individual characters. Instead, common parts of
words can be represented as larger subword tokens.

This can make character-level tasks such as reversing words difficult. A
language model primarily processes sequences of tokens rather than explicit
sequences of individual characters, so manipulating characters inside a
multi-character token is less direct.

Other tokenisation-sensitive prompts include counting the letters in a word,
identifying the nth character, or inserting a symbol between every character.

### Tokenisation and multi-linguality

Many NLP systems and the tokenisers used with them are primarily trained on English data. In the next task, you will reflect on the effect this has when they are used to process non-English data.

The *context length* of a language model is the maximum number of preceding tokens the model can condition on when predicting the next token. This number is fixed and cannot be changed after training the model. For example, the context length of GPT-2 ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)) is 1,024. 

While the context length of a language model is fixed, the amount of information that can be squeezed into this context length will depend on the tokeniser. Informally speaking, a model that needs more tokens to represent a given text cannot extract as much information from that text as one that needs fewer tokens.

#### 🎓 Task 1.06: Tokenisation and multi-linguality

Train a tokeniser on the English text file from Task&nbsp;1.04 (or use the provided one) and apply it to the same text. How many tokens does it split the text into? Based on this, what is the expected number of Unicode characters of English text that can be fit into the GPT-2 context window?

What do the numbers look like if you apply the English tokeniser to the Icelandic text instead? How do you explain the differences?

Interpreting the expected number of Unicode characters as a measure of representation efficiency, what do your results tell you about the efficiency of a language model primarily trained on English data when it is used to process non-English data? Relate these findings to the issue of tokeniser fairness.

In [17]:
# Read the English Wikipedia text.
with open("wiki-en-1m.txt", "r", encoding="utf-8") as f:
    text_en = f.read()

# Read the Icelandic Wikipedia text.
with open("wiki-is-1m.txt", "r", encoding="utf-8") as f:
    text_is = f.read()

# Apply the English-trained tokenizer to both texts.
tokens_en = tok.encode(text_en)
tokens_is = tok.encode(text_is)

# Print the number of Unicode characters and resulting tokens.
print("English characters:", len(text_en))
print("English tokens:", len(tokens_en))

print("Icelandic characters:", len(text_is))
print("Icelandic tokens:", len(tokens_is))

English characters: 1000000
English tokens: 379779
Icelandic characters: 1000000
Icelandic tokens: 754866


In [18]:
# Average number of characters represented by one token.
chars_per_token_en = len(text_en) / len(tokens_en)
chars_per_token_is = len(text_is) / len(tokens_is)

# GPT-2 has a context length of 1024 tokens.
context_size = 1024

# Expected number of Unicode characters that fit in the context window.
expected_chars_en = context_size * chars_per_token_en
expected_chars_is = context_size * chars_per_token_is

print("English characters per token:", chars_per_token_en)
print("Icelandic characters per token:", chars_per_token_is)

print("Expected English characters in 1024 tokens:", expected_chars_en)
print("Expected Icelandic characters in 1024 tokens:", expected_chars_is)

English characters per token: 2.63311030889017
Icelandic characters per token: 1.324738430396918
Expected English characters in 1024 tokens: 2696.3049563035343
Expected Icelandic characters in 1024 tokens: 1356.532152726444


### ANSWER Task 1.06

When the English-trained tokenizer is applied to the 1,000,000-character
English Wikipedia text, it produces **379,779 tokens**. This corresponds to
approximately **2.63 Unicode characters per token**.

Since GPT-2 has a context length of 1,024 tokens, the expected amount of
English text that can fit into the context window is approximately

`1024 × 2.63 ≈ 2696 characters`.

When the same English-trained tokenizer is applied to the 1,000,000-character
Icelandic Wikipedia text, it produces **754,866 tokens**. This corresponds to
only approximately **1.32 Unicode characters per token**.

The expected amount of Icelandic text that fits into the same 1,024-token
context window is therefore approximately

`1024 × 1.32 ≈ 1357 characters`.

The difference occurs because the tokenizer was trained on English text. Its
merge rules are therefore adapted to frequent English byte and subword
patterns, allowing common English sequences to be represented by relatively
few tokens. Icelandic contains different character combinations, word forms
and characters that are less well represented by these learned merge rules,
so the text is split into many more tokens.

This shows that a fixed token context length does not correspond to a fixed
amount of textual information across languages. In this experiment, the same
1,024-token context can contain roughly twice as many English characters as
Icelandic characters. Therefore, a language model using an English-oriented
tokenizer can represent English text more efficiently than Icelandic text.

This relates to **tokeniser fairness** because users of languages that are less
well represented in tokenizer training data effectively receive a smaller
usable context window and require more tokens to represent the same amount of
text. This can make processing such languages less efficient even when the
underlying language model has the same nominal context length.

## Part 2: Embeddings

In the second part of the lab, you will explore embeddings. An embedding layer is a network component that assigns each item in a finite set of elements (often called a *vocabulary*) a fixed-size vector. At first, these vectors are filled with random values, but during training, they are adjusted to suit the task at hand.

### Bag-of-words classifier

To help you build an intuition for embeddings and the vector representations learned by them, we will use a simple bag-of-words text classifier. The core part of this classifier only takes a few lines of code:

In [7]:
import torch.nn as nn


class Classifier(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.linear = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        return self.linear(self.embedding(x).mean(dim=-2))

#### 🧩 Task 1.07: Bag-of-words classifier

Explain how the bag-of-words classifier works. How does the code match the diagram you saw in the lectures? Why is there only one `nn.Embedding`, while the diagram shows three embedding layers? What does the keyword argument `dim=-2` do?


### ANSWER

The classifier first maps each token ID in the input to a dense vector using
`nn.Embedding`. If the input has shape

`[batch_size, sequence_length]`

then after the embedding layer it has shape

`[batch_size, sequence_length, embedding_dim]`.

The expression

`self.embedding(x).mean(dim=-2)`

takes the mean of all token embeddings in each review (Amazon customer review for our purposes). Since `-2` refers to
the second-to-last dimension, this averages over the sequence-length dimension.
The result therefore has shape

`[batch_size, embedding_dim]`.

This produces one fixed-size vector for the whole review, independently of
the review length. Because the embeddings are averaged, the model ignores
word order and only captures which words occur in the review, which is why it
is a bag-of-words model.

The averaged review representation is then passed through

`nn.Linear(embedding_dim, num_classes)`

which maps the review vector to one score for each output class. These scores
can later be used with a classification loss such as cross-entropy.

Although the lecture diagram may show several embedding boxes, the code only
needs one `nn.Embedding` layer because the same embedding lookup table is
shared by all token positions. Each input token is simply used as an index
into the same embedding matrix.

### Dataset

You will apply the classifier to a small dataset with Amazon customer reviews. This dataset is taken from [a much larger dataset](https://www.cs.jhu.edu/~mdredze/datasets/sentiment/) first described by [Blitzer et al. (2007)](https://aclanthology.org/P07-1056/).

The dataset contains whitespace-tokenised product reviews from two categories: cameras (`camera`) and music (`music`). Each review is additionally annotated for sentiment towards the product at hand: negative (`neg`) or positive (`pos`). The category and sentiment labels are prepended to the review. As an example, here is the first review from the training data:

```
music neg oh man , this sucks really bad . good thing nu-metal is dead . thrash metal is real metal , this is for posers
```

The next cell contains a custom [`Dataset`](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) class for the review dataset. To initialise an instance of this class, you specify the name of the file containing the reviews you want to load (`filename`) and which of the two labels you want to use (`label`): product category (0) or sentiment (1).

In [19]:
from torch.utils.data import Dataset


class ReviewDataset(Dataset):
    def __init__(self, filename: str, label: int = 0) -> None:
        with open(filename) as f:
            tokenized_lines = [line.split() for line in f]
        self.items = [(tokens[2:], tokens[label]) for tokens in tokenized_lines]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[list[str], str]:
        return self.items[idx]

### Vectoriser

To feed a review into the bag-of-words classifier, you first need to turn it into a vector of token IDs. Likewise, you need to convert the label (product category or sentiment) into an integer. The next cell contains a partially completed `ReviewVectoriser` class that handles this transformation.

In [22]:
from collections import Counter

import torch

# Type abbreviation for review–label pairs
Item = tuple[list[str], str]


class ReviewVectorizer:
    PAD = "[PAD]"
    UNK = "[UNK]"

    def __init__(self, dataset: ReviewDataset, n_vocab: int = 1024) -> None:
        # Unzip the dataset into two tuples:
        # one containing all reviews and one containing all labels.
        reviews, labels = zip(*dataset)

        # Count how often each token appears across all reviews.
        counter = Counter(t for r in reviews for t in r)

        # Keep the n_vocab - 2 most common tokens.
        # Two vocabulary positions are reserved for [PAD] and [UNK].
        most_common = [t for t, _ in counter.most_common(n_vocab - 2)]

        # Map each token to a unique integer ID.
        # [PAD] gets ID 0, [UNK] gets ID 1.
        self.t2i = {
            t: i
            for i, t in enumerate([self.PAD, self.UNK] + most_common)
        }

        # Map each unique label to a unique integer ID.
        # Sorting makes the mapping deterministic.
        self.l2i = {
            l: i
            for i, l in enumerate(sorted(set(labels)))
        }

    def __call__(
        self,
        items: list[Item]
    ) -> tuple[torch.Tensor, torch.Tensor]:

        # Separate the batch into reviews and labels.
        reviews, labels = zip(*items)

        # Find the length of the longest review in this batch.
        # All shorter reviews must be padded to this length.
        max_len = max(len(review) for review in reviews)

        # Get the integer IDs of the special tokens.
        pad_id = self.t2i[self.PAD]
        unk_id = self.t2i[self.UNK]

        # This will contain one vector of token IDs per review.
        x = []

        for review in reviews:
            # Convert each token to its ID.
            # If the token is not in the vocabulary, use [UNK].
            review_ids = [
                self.t2i.get(token, unk_id)
                for token in review
            ]

            # Add [PAD] tokens so that every review in the batch
            # has the same length.
            review_ids += [pad_id] * (max_len - len(review_ids))

            x.append(review_ids)

        # Convert the string labels into integer IDs.
        y = [self.l2i[label] for label in labels]

        # Convert the Python lists into PyTorch tensors.
        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long),
        )

A `ReviewVectoriser` maps tokens and labels to IDs using two Python dictionaries. These dictionaries are set up when the vectoriser is initialised and queried when the vectoriser is called on a batch of review–label pairs. They include IDs for two special tokens:

`[PAD]` (Padding): Reviews can have different lengths, but PyTorch requires all vectors in a batch to be the same size. To handle this, the vectoriser adds `[PAD]` tokens to the end of shorter reviews so they match the length of the longest review in the batch.

`[UNK]` (Unknown): If a review contains a token that is not in the token-to-ID dictionary, the vectoriser assigns it the ID of the `[UNK]` token instead of a regular ID.

#### 🎓 Task 1.08: Vectoriser

Explain and complete the code of the vectoriser. Follow these steps:

**Step&nbsp;1.** Explain how unzipping works. What are the types of `reviews` and `labels`?

**Step&nbsp;2.** Explain how the token-to-ID and label-to-ID mappings are constructed. How does the `most_common()` method deal with elements that occur equally often?

**Step&nbsp;3.** Complete the implementation of the `__call__()` method. This method should convert a list of $m$ review–label pairs into a pair $(X, y)$ where $X$ is a matrix containing the vectors with token IDs for the reviews, and $y$ is a vector containing the IDs of the corresponding labels.


### ANSWER

`zip(*dataset)` unzips the dataset into two tuples. `reviews` contains all
review token lists, while `labels` contains all corresponding string labels.

The token-to-ID mapping `t2i` is built by first counting all tokens in the
training data and keeping the most frequent `n_vocab - 2` tokens. Two IDs are
reserved for the special tokens `[PAD]` and `[UNK]`. The label-to-ID mapping
`l2i` assigns an integer ID to each unique label.

`Counter.most_common()` returns items in descending frequency order. For items
with equal counts, their order follows the order in which they were first
encountered.

When the vectoriser is called on a batch, each token is replaced by its token
ID. Unknown tokens are mapped to `[UNK]`. Since reviews can have different
lengths, shorter reviews are padded with `[PAD]` until they match the longest
review in the batch. The result is a matrix `X` of token IDs and a vector `y`
of label IDs, both returned as PyTorch tensors.

### Training the classifier

With the vectoriser completed, you are ready to train a classifier. More specifically, you can train two separate classifiers: one to predict the product category of a review, and one to predict the sentiment. The next cell contains a simple training loop that you can adapt for this purpose.

In [48]:
import torch.nn.functional as F


def train(
    filename: str = "reviews-train.txt",
    label: int = 0,
    n_vocab: int = 1024,
    embedding_dim: int = 64,
    learning_rate: float = 0.001,
    batch_size: int = 16,
    epochs: int = 50,
    kaiming_init: bool = False,
):
    """
    Train the bag-of-words classifier.

    Parameters correspond to the previously hard-coded values,
    but can now be changed when calling the function.
    """

    # Load the review dataset.
    # label=0 means product category classification,
    # while label=1 means sentiment classification.
    dataset = ReviewDataset(filename, label=label)

    # Build the vectoriser from the training dataset.
    # It creates the token and label vocabularies.
    processor = ReviewVectorizer(dataset, n_vocab)

    # Create the classifier.
    # - n_vocab: number of possible token IDs
    # - embedding_dim: size of each learned token embedding
    # - len(processor.l2i): number of output classes
    model = Classifier(
    n_vocab,
    embedding_dim,
    len(processor.l2i),
    kaiming_init=kaiming_init,
    )

    # Adam updates the model parameters using the computed gradients.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    # DataLoader creates mini-batches from the dataset.
    # shuffle=True changes the example order each epoch.
    # collate_fn=processor converts reviews and labels into tensors.
    data_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=processor,
    )

    # Repeat training for the requested number of epochs.
    for epoch in range(epochs):

        # Put the model into training mode.
        model.train()

        # Accumulate the losses so we can report
        # the average loss for the epoch.
        running_loss = 0

        # bx contains a batch of review token IDs.
        # by contains the corresponding label IDs.
        for bx, by in data_loader:

            # Remove gradients left over from the previous batch.
            optimizer.zero_grad()

            # Forward pass:
            # compute the model's class scores for each review.
            output = model(bx)

            # Compare the predicted class scores with
            # the correct labels using cross-entropy loss.
            loss = F.cross_entropy(output, by)

            # Backward pass:
            # compute gradients of the loss with respect
            # to all trainable model parameters.
            loss.backward()

            # Update the model parameters using the gradients.
            optimizer.step()

            # Store the scalar loss value for reporting.
            running_loss += loss.item()

        # Print the average loss over all batches in this epoch.
        print(
            f"Epoch {epoch}, "
            f"loss: {running_loss / len(data_loader):.4f}"
        )

    # Return both the fitted vectoriser and trained model.
    return processor, model

#### 🎓 Task 1.09: Training loop

Explain the training loop. Follow these steps:

**Step&nbsp;1.** Go through the training loop line-by-line and add comments where you find it suitable. Your comments should be detailed enough for you to explain the main steps of the loop.

**Step&nbsp;2.** The training loop contains various hard-coded values like filename, learning rate, batch size, and epoch count. This makes the code less flexible. Revise the code so that you can specify these values using keyword arguments. Use the concrete values from the code as defaults.


### REMARK

The training function first loads the dataset and constructs a
`ReviewVectorizer`, which converts reviews and labels into integer IDs. It
then creates the bag-of-words classifier and an Adam optimizer.

A `DataLoader` divides the dataset into mini-batches, shuffles the training
examples, and uses the vectoriser as its `collate_fn` to convert each batch
into tensors.

For every batch, `optimizer.zero_grad()` clears gradients from the previous
update. The forward pass `model(bx)` produces class scores for each review.
`F.cross_entropy(output, by)` compares these scores with the correct labels
and produces a loss value. `loss.backward()` computes gradients of the loss
with respect to the model parameters, and `optimizer.step()` updates those
parameters.

This process is repeated for all batches and for multiple epochs. The average
training loss is printed after each epoch.

The revised function exposes the filename, label type, vocabulary size,
embedding dimension, learning rate, batch size, and number of epochs as
keyword arguments, while keeping the original values as defaults.

#### 🧩 Task 1.10: Training the classifier

Adapt the next cell to train the classifier for the two prediction tasks. Based on the loss values, which task appears to be the harder one? What is the purpose of setting a seed?

In [49]:
torch.manual_seed(42)

print("Product category classification:")
category_vectorizer, category_model = train(label=0)

# Reset the seed so that the second experiment also starts
# from the same reproducible random state.
torch.manual_seed(42)

print("\nSentiment classification:")
sentiment_vectorizer, sentiment_model = train(label=1)

Product category classification:
Epoch 0, loss: 0.6838
Epoch 1, loss: 0.6486
Epoch 2, loss: 0.6046
Epoch 3, loss: 0.5415
Epoch 4, loss: 0.4708
Epoch 5, loss: 0.4008
Epoch 6, loss: 0.3403
Epoch 7, loss: 0.2900
Epoch 8, loss: 0.2518
Epoch 9, loss: 0.2230
Epoch 10, loss: 0.1983
Epoch 11, loss: 0.1775
Epoch 12, loss: 0.1592
Epoch 13, loss: 0.1458
Epoch 14, loss: 0.1322
Epoch 15, loss: 0.1226
Epoch 16, loss: 0.1132
Epoch 17, loss: 0.1068
Epoch 18, loss: 0.0973
Epoch 19, loss: 0.0913
Epoch 20, loss: 0.0869
Epoch 21, loss: 0.0808
Epoch 22, loss: 0.0758
Epoch 23, loss: 0.0710
Epoch 24, loss: 0.0689
Epoch 25, loss: 0.0650
Epoch 26, loss: 0.0609
Epoch 27, loss: 0.0579
Epoch 28, loss: 0.0550
Epoch 29, loss: 0.0522
Epoch 30, loss: 0.0499
Epoch 31, loss: 0.0479
Epoch 32, loss: 0.0460
Epoch 33, loss: 0.0438
Epoch 34, loss: 0.0415
Epoch 35, loss: 0.0394
Epoch 36, loss: 0.0383
Epoch 37, loss: 0.0366
Epoch 38, loss: 0.0354
Epoch 39, loss: 0.0340
Epoch 40, loss: 0.0326
Epoch 41, loss: 0.0316
Epoch 42, l

#### ANSWER

Based on the embeddings Sentiment classificatio seems harder than Category classification.

### Inspecting the embeddings

Now that you have trained the classifier on two separate prediction tasks, it is interesting to inspect and compare the embedding vectors it learned in the process. For this you will use an online tool called the [Embedding Projector](http://projector.tensorflow.org). The next cell contains code to save the embeddings from a trained classifier in a format that can be loaded into this tool.

In [44]:
def save_embeddings(
    vectorizer: ReviewVectorizer,
    model: Classifier,
    vectors_filename: str,
    metadata_filename: str,
):
    i2t = {i: t for t, i in vectorizer.t2i.items()}
    embeddings = model.embedding.weight.detach().numpy()
    items = [(i2t[i], e) for i, e in enumerate(embeddings)]
    with open(vectors_filename, "wt") as f1, open(metadata_filename, "wt") as f2:
        for w, e in items:
            print("\t".join("{:.5f}".format(x) for x in e), file=f1)
            print(w, file=f2)

Call this code as follows:

In [45]:
# Save embeddings learned for product category classification
save_embeddings(
    category_vectorizer,
    category_model,
    "category_vectors.tsv",
    "category_metadata.tsv",
)

# Save embeddings learned for sentiment classification
save_embeddings(
    sentiment_vectorizer,
    sentiment_model,
    "sentiment_vectors.tsv",
    "sentiment_metadata.tsv",
)

#### 🎓 Task 1.11: Inspecting the embeddings

Load the embeddings from the two classification tasks (product category classification and sentiment classification) into the Embedding Projector web app and inspect the vector spaces. How do they compare visually? Does the visualisation make sense to you?

The Embedding Projector offers visualisations based on three dimensionality reduction methods: [UMAP](https://umap-learn.readthedocs.io/en/latest/), [T-SNE](https://en.m.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding), and [PCA](https://en.m.wikipedia.org/wiki/Principal_component_analysis). Which of these seems most useful to you?

Focus on the embeddings for the words *repair* and *sturdy*. Are they close to each other or far away from another? What happens if you switch to the other task? How do you explain that?

### REMARKS

I visualised the embeddings learned for both product-category classification
and sentiment classification using PCA, T-SNE, and UMAP.

Visually, I did not observe a very strong or obvious difference between the
two embedding spaces. Likewise, none of the three dimensionality-reduction
methods produced a dramatically clearer structure than the others. This is
reasonable because the original embeddings are 64-dimensional, while the
visualisations reduce them to only 3 dimensions, a lot of the structure
can be lost during projection.

Among the three methods, I found **UMAP** slightly more useful because it gave
a reasonably interpretable view of both local neighbourhoods and the overall
structure, although the differences compared with T-SNE and PCA were not very
large in this experiment.

The words `repair` and `sturdy` appeared relatively far apart in both the
category-classification and sentiment-classification embedding spaces. This
suggests that the two words did not acquire very similar representations for
either task.

The embeddings are task-dependent: they are trained only to help solve the
classification objective, rather than to reproduce general semantic
similarity between words. Therefore, words that may seem related to a human
do not necessarily have to be close in the learned vector space. In addition,
the simplicity of the bag-of-words model and the dimensionality reduction used
for visualisation can make differences between the two trained embedding spaces
hard to observe directly.

### Initialisation of embedding layers

The error surfaces explored when training neural networks can be very complex. Because of this, it is crucial to choose “good” initial values for the parameters. In the final task of this lab, you will run a small experiment to see how alternative initialisations can affect a model’s performance.

In PyTorch, the weights of the embedding layer are initially set by sampling from the standard normal distribution, $\mathcal{N}(0, 1)$. However, research suggests other approaches may work better. For example, given that embedding layers share similarities with linear layers, it makes sense to use the same initialisation method for both. The default initialisation method for linear layers in PyTorch is the so-called Kaiming initialisation, introduced by [He et al. (2015)](https://www.cv-foundation.org/openaccess/content_iccv_2015/papers/He_Delving_Deep_into_ICCV_2015_paper.pdf).

#### 🧩 Task 1.12: Initialisation of embedding layers

Check the [source code of `nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear) to see how PyTorch initialises the weights of linear layers using the Kaiming initialisation method. Apply the same method to the embedding layer of your classifier and see how this affects the loss of your model and the vector spaces.

In [50]:
import math
import torch.nn as nn


class Classifier(nn.Module):
    def __init__(
        self,
        num_embeddings,
        embedding_dim,
        num_classes,
        kaiming_init=False,
    ):
        super().__init__()

        # Embedding layer: maps each token ID to a learned vector.
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)

        # Linear layer: maps the averaged review embedding
        # to one score for each class.
        self.linear = nn.Linear(embedding_dim, num_classes)

        # Optionally initialize the embedding weights using
        # the same Kaiming uniform initialization as nn.Linear.
        if kaiming_init:
            nn.init.kaiming_uniform_(
                self.embedding.weight,
                a=math.sqrt(5),
            )

    def forward(self, x):
        # Look up token embeddings, average them across the review,
        # and produce the class scores.
        return self.linear(self.embedding(x).mean(dim=-2))


# Fix the random seed for reproducibility.
torch.manual_seed(42)

# Train the sentiment classifier using Kaiming initialization.
kaiming_vectorizer, kaiming_model = train(
    label=1,
    kaiming_init=True,
)

# Save the learned embeddings for inspection in Embedding Projector.
save_embeddings(
    kaiming_vectorizer,
    kaiming_model,
    "kaiming_vectors.tsv",
    "kaiming_metadata.tsv",
)

Epoch 0, loss: 0.6915
Epoch 1, loss: 0.6817
Epoch 2, loss: 0.6580
Epoch 3, loss: 0.6171
Epoch 4, loss: 0.5678
Epoch 5, loss: 0.5207
Epoch 6, loss: 0.4774
Epoch 7, loss: 0.4402
Epoch 8, loss: 0.4106
Epoch 9, loss: 0.3862
Epoch 10, loss: 0.3663
Epoch 11, loss: 0.3450
Epoch 12, loss: 0.3264
Epoch 13, loss: 0.3126
Epoch 14, loss: 0.3005
Epoch 15, loss: 0.2876
Epoch 16, loss: 0.2760
Epoch 17, loss: 0.2660
Epoch 18, loss: 0.2604
Epoch 19, loss: 0.2464
Epoch 20, loss: 0.2376
Epoch 21, loss: 0.2305
Epoch 22, loss: 0.2260
Epoch 23, loss: 0.2149
Epoch 24, loss: 0.2104
Epoch 25, loss: 0.2017
Epoch 26, loss: 0.1969
Epoch 27, loss: 0.1917
Epoch 28, loss: 0.1857
Epoch 29, loss: 0.1820
Epoch 30, loss: 0.1742
Epoch 31, loss: 0.1723
Epoch 32, loss: 0.1665
Epoch 33, loss: 0.1607
Epoch 34, loss: 0.1598
Epoch 35, loss: 0.1542
Epoch 36, loss: 0.1471
Epoch 37, loss: 0.1436
Epoch 38, loss: 0.1415
Epoch 39, loss: 0.1379
Epoch 40, loss: 0.1349
Epoch 41, loss: 0.1316
Epoch 42, loss: 0.1271
Epoch 43, loss: 0.123

### ANSWER Task 1.12

By default, PyTorch initialises the weights of `nn.Embedding` by sampling
from a standard normal distribution, $\mathcal{N}(0,1)$. The `nn.Linear`
layer instead uses Kaiming uniform initialisation, implemented using
`kaiming_uniform_` with `a = sqrt(5)`.

I applied the same initialisation to the embedding layer using

`nn.init.kaiming_uniform_(self.embedding.weight, a=math.sqrt(5))`.

I then trained the sentiment classifier for 50 epochs using the same settings
and random seed as the original experiment.

With the default embedding initialisation, the training loss decreased from
approximately **0.6932** to **0.2181**. With Kaiming initialisation, it
decreased from approximately **0.6915** to **0.1083**.

Therefore, Kaiming initialisation improved the optimisation in this experiment.
The loss decreased faster and reached a substantially lower value after the
same number of training epochs. This demonstrates that the initial values of
the embedding vectors can have an important effect on the optimisation path
and the final parameters learned by the model.

When inspecting the resulting embeddings in the Embedding Projector, any
differences in the vector space can be explained by the fact that changing
the initialisation changes the starting point of optimisation and can therefore
lead the model to learn a different embedding space.

**🥳 Congratulations on finishing lab&nbsp;1!**